In [5]:
import sys

sys.path.append("..")
import datetime
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from tqdm import tqdm

from src import request_handler

In [6]:
# DATE = datetime.datetime.now()
# comment out line above and use line below if you want to fetch the last data
DATE = datetime.datetime.fromisoformat(
    "2026-05-02"
)  # Input data in the <YYYY-MM-DD> format in the string

PARENT_PATH = Path.cwd().parent

## Load the FIRMS api key 
Set the api key from the .env file as environment variable, then read the it from there. I avoid to push the api key in the remote repo,  and this approach also offer the possibility to store and read the api key from github secrets if using github workflows (e.g. to deploy the map on github pages)

In [7]:
load_dotenv()  # sets variables in os.environ from .env file

API_KEY = os.getenv("FIRM_API_KEY")
if not API_KEY:
    raise Exception("Missing FIRM_API_KEY. Set it in your environment or .env file.")

### Test api key
---

In [8]:
url = "https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY="
parameters = {"MAP_KEY": API_KEY}

response = request_handler.handle_get_request(url, parameters)
print(f"Api request successful\n{'-' * 22}")

# Convert the JSON response into a Python dictionary
data = response.json()
print(f"Current transactions: {data['current_transactions']}/5000")

Api request successful
----------------------
Current transactions: 0/5000


## Create the `data` directory and its children directories if not present
Those directories serve to save the downloaded raw data and the later processed data

In [9]:
dir_path = PARENT_PATH.joinpath("data")
if not dir_path.exists():
    Path.mkdir(dir_path)
    Path.mkdir(dir_path.joinpath("raw"))
    Path.mkdir(dir_path.joinpath("processed"))

## Download the necessary data

### Fetch VIIRS NRT active fire data from the last 14 days using the FIRMS api
---

In [ ]:
date = DATE + datetime.timedelta(days=-1)
data = []

for _i in tqdm(range(0, 7)):
    date_str = date.strftime("%Y-%m-%d")
    date = date + datetime.timedelta(days=-2)
    url = (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        + API_KEY
        + "/VIIRS_SNPP_NRT/world/2/"
        + date_str
    )

    response = request_handler.handle_get_request(url)
    parsed_response = response.content.decode("utf-8")
    # if data is still empty, also append the column
    if not data:
        data.extend(parsed_response.split("\n"))
    else:
        data.extend(parsed_response.split("\n")[1:])

    # api seems to behave well, but we sleep half second nonetheless
    time.sleep(0.5)

# use Path object so we don't have to worry about path separator differences
file_path = PARENT_PATH.joinpath(
    "data/raw", f"VIIRS_SNNP_NRT_world_14days_{DATE.strftime('%Y-%m-%d')}.csv"
)

with open(file_path, "w") as file:
    file.write("\n".join(data))


 43%|████▎     | 3/7 [00:06<00:08,  2.16s/it]

### Download world countries' boundaries geometries
---

In [ ]:
url = "https://github.com/wmgeolab/geoBoundaries/raw/refs/heads/main/releaseData/CGAZ/geoBoundariesCGAZ_ADM0.geojson"
response = request_handler.handle_get_request(url)
file_path = PARENT_PATH.joinpath("data/raw", "geoboundaries_world.geojson")
with open(file_path, "wb") as f:
    f.write(response.content)

### Download countries' surface area dataset
---

In [ ]:
url = "https://data360files.worldbank.org/data360-data/data/WB_WDI/WB_WDI_AG_SRF_TOTL_K2.csv"
response = request_handler.handle_get_request(url)
file_path = PARENT_PATH.joinpath("data/raw", "surface_area.csv")
with open(file_path, "wb") as f:
    f.write(response.content)